# YOLO11n N1 R1-R2 — resume smoke to epoch 2

Train/validation only. Classifiers are not retrained. Never opens the sealed test.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

TRACK = 'n1'
MODE = 'resume'
STOP_AFTER = 2
REPO_URL = 'https://github.com/ScottPhantom/Traffic_SIgn_Detection.git'
SOURCE_COMMIT = '0ec545183bf29de035f440bb0307842ae132f4ca'
SOURCE_QA_SHA256 = '5fd51deea1679263b224fc45531f30a3d0a5aac86b3d044d262446e77a48b92a'
WEIGHT_URL = 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n.pt'
WEIGHT_SHA256 = '0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1'
DRIVE_ROOT = Path('/content/drive/MyDrive/CPV301/Traffic_Sign_Detector')
STATE = DRIVE_ROOT / 'M1_D1_Colab/runs/yolo11n_n1_seed_20261018_r1_r1'
REPO = Path('/content/Traffic_Sign_Detector')
assert MODE in {'fresh', 'resume'}
assert (MODE, STOP_AFTER) in {('fresh', 1), ('resume', 2)}

ARCHIVE = DRIVE_ROOT / 'M1_D1_Colab/yolo11n_n1_r1.tar.gz'
ARCHIVE_SHA256 = '5798ad93f552e1f7c36b46a3470fc9f8f85c5125df753786d63e228822299d05'
MANIFEST_SHA256 = '8e6ffe7761b78ba4e44a86cd2caa9a3a155ba0874cbe39bb20669af3773bb6da'
INPUT_ROOT = Path('/content/yolo11n_n1_input')
SOURCE_QA = REPO / 'docs/review/yolo11n_n1_r1_r2_source_independent_qa_record.md'


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys

def file_hash(path):
    value = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()

gpu_name = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert gpu_name == 'NVIDIA L4', gpu_name
assert len(SOURCE_COMMIT) == 40
assert len(SOURCE_QA_SHA256) == 64
assert len(WEIGHT_SHA256) == 64
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO, check=True)
observed = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True,
    capture_output=True, text=True,
).stdout.strip()
assert observed == SOURCE_COMMIT
assert not subprocess.run(
    ['git', 'status', '--porcelain'], cwd=REPO, check=True,
    capture_output=True, text=True,
).stdout.strip()
if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.11.14'], check=True)
print({'gpu': gpu_name, 'checkout': observed, 'track': TRACK, 'mode': MODE})


In [ ]:
import socket

assert ARCHIVE.is_file() and not ARCHIVE.is_symlink()
assert file_hash(ARCHIVE) == ARCHIVE_SHA256
if INPUT_ROOT.exists():
    shutil.rmtree(INPUT_ROOT)
subprocess.run([
    sys.executable, str(REPO / 'detector/colab/extract_cascade_b0_r1.py'),
    str(ARCHIVE), str(INPUT_ROOT),
    '--archive-sha256', ARCHIVE_SHA256,
    '--manifest-sha256', MANIFEST_SHA256,
], check=True)
assert SOURCE_QA.is_file() and file_hash(SOURCE_QA) == SOURCE_QA_SHA256
manifest = json.loads((INPUT_ROOT / 'manifest.json').read_text())
assert manifest['package'] == 'YOLO11N-N1-R1'
assert manifest['test_included'] is False
assert manifest['sealed_test_accessed'] is False
assert manifest['pretrained_weight_sha256'] == WEIGHT_SHA256
if MODE == 'fresh':
    assert not STATE.exists(), STATE
else:
    assert (STATE / 'detector/recovery/latest.json').is_file()
    prior_execution = json.loads((STATE / 'colab_execution.json').read_text())
    prior_attempts = prior_execution['execution_attempts']
    assert len(prior_attempts) == 1 and prior_attempts[0]['mode'] == 'fresh'
    boot_id = Path('/proc/sys/kernel/random/boot_id').read_text()
    current_runtime = hashlib.sha256((socket.gethostname() + boot_id).encode()).hexdigest()
    assert current_runtime != prior_attempts[0]['runtime_digest'], 'A new L4 runtime is required'
subprocess.run(['uv', 'sync', '--frozen'], cwd=REPO / 'detector', check=True)
print({'archive_sha256': ARCHIVE_SHA256, 'manifest_sha256': MANIFEST_SHA256})


In [ ]:
command = [
    'uv', 'run', 'python', str(REPO / 'detector/colab/run_yolo11n_n1_r1_r1.py'),
    'detector', '--root', str(INPUT_ROOT), '--state', str(STATE),
    '--manifest-sha256', MANIFEST_SHA256,
    '--source-qa-record', str(SOURCE_QA), '--source-qa-sha256', SOURCE_QA_SHA256,
    '--mode', MODE, '--stop-after', str(STOP_AFTER),
]
environment = dict(os.environ)
environment['PYTHONPATH'] = str(REPO / 'detector')
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO / 'detector', env=environment, check=True)


In [ ]:
execution = json.loads((STATE / 'colab_execution.json').read_text())
latest = json.loads((STATE / 'detector/recovery/latest.json').read_text())
assert execution['sealed_test_accessed'] is False
assert execution['status'] == 'SMOKE_STOP'
assert latest['generation'] == f'epoch-{STOP_AFTER:04d}'
attempts = execution['execution_attempts']
assert attempts[-1]['mode'] == MODE
if MODE == 'fresh':
    assert len(attempts) == 1
else:
    assert len(attempts) == 2
    assert attempts[0]['runtime_digest'] != attempts[1]['runtime_digest']
print({'status': 'SMOKE_PASS', 'track': TRACK, 'epoch': STOP_AFTER, 'seal': False})
